# A. 데이터 확인 및 분할 파일 생성
녹화 21개 복원 → 8종 통합 → `splits/folds.json` 저장 → 특성 A~C 확인

In [ ]:
!git clone https://github.com/ekgP525/ML-imu.git
%cd ML-imu
!pip install -q -r requirements.txt
!python data/download.py

In [ ]:
import sys; sys.path.append('src')
import numpy as np
from recordings import load_raw, check_within_group_order, restore_recordings, load_dataset, summary
from splits import make_folds, make_material_out_folds, save_folds, load_folds, fold_indices
from features import extract_features

## 1. 녹화 복원 확인
- 그룹 안 순서: 최댓값이 아주 작아야 함
- 경계: 같은 재질로 이어지는 59곳은 ≤ 0.0014, 재질이 바뀌는 20곳은 0.10~2.00

In [ ]:
X0, lab, grp = load_raw('data')
print(X0.shape, lab[:3], grp[:3])
print(check_within_group_order(X0, grp))
rec, bnd = restore_recordings(X0, lab, grp, return_boundaries=True)
print(bnd.groupby(['same_surface', 'joined'])['qdist'].agg(['count', 'min', 'max']))
print('녹화 수:', len(np.unique(rec)))

## 2. 8종 통합 + 재질별 표 (README 표와 같아야 함)

In [ ]:
X, y, rec = load_dataset('data')
print(X.shape)
summary(y, rec)

## 3. 분할 파일 저장

In [ ]:
folds = make_folds(np.unique(rec))
material_folds = make_material_out_folds(y, rec)
save_folds(folds, material_folds=material_folds)
print(len(folds), len(material_folds))
tr, te = fold_indices(load_folds()[0], rec)
print(len(tr), len(te))

## 4. 특성 A~C 확인

In [ ]:
F = extract_features(X)
print(F.shape)
F.describe().T.head(12)